In [109]:
import matplotlib
import numpy as np
import pandas as pd
from narwhals import Int64
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, brier_score_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import matplotlib.pyplot as plt
from sklearn.base import clone

In [100]:
train_df = pd.read_csv(r"/data/cleaned_data/stop_search_train1.csv", index_col=0)
test_df = pd.read_csv(r"/data/cleaned_data/stop_search_test1.csv", index_col=0)

# Preprocessing before pipeline and modeling
## We drop unnecessary columns, define categorical and discrete/numerical columns for imputation, and convert categorical variables into strings

In [101]:
# Dropping unnecessary columns for modeling
train_date = train_df["event_dt"]
train_df = train_df.drop(["event_dt", "orig_age", "vital_flag", "orig_height", "orig_weight", "BMI", "year", "Zone", "Latitude", "Longitude", "SubjectID"], axis = 1)

# Dropping unnecessary columns for modeling
test_date = test_df["event_dt"]
test_df = test_df.drop(["event_dt", "orig_age", "vital_flag", "orig_height", "orig_weight", "BMI", "year", "Zone", "Latitude", "Longitude", "SubjectID"], axis = 1)

# Unfloating zip code
train_df["Zip"] = train_df["Zip"].astype("Int64")
test_df["Zip"] = test_df["Zip"].astype("Int64")

# Defining categorical columns
cat_cols = ["SubjectRace", "Zip", "District"]
# Defining discrete/numerical columns
dis_cols = ["SubjectGender", "SubjectAge", "SubjectHeight", "SubjectWeight", "isLA","VehicleYear", "SubjectHasPhotoID", "hour", "night", "isDriver"]
# Numerical columns we would like to scale
scale_cols = ["SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear", "hour"]
# Binary columns
bin_cols = ["SubjectGender", "isLA", "SubjectHasPhotoID", "night", "isDriver"]

# Converting variables into strings
train_df[cat_cols] = train_df[cat_cols].astype(str)
# Converting variables into strings
test_df[cat_cols] = test_df[cat_cols].astype(str)

In [103]:
target = "search_occurred"
X_train = train_df.drop([target], axis = 1)[cat_cols + dis_cols]
y_train = train_df[target]
X_test = test_df.drop([target], axis = 1)[cat_cols + dis_cols]
y_test = test_df[target]

# Defining classes to build pipeline
## CatBinner groups sparse labels in categorical variables in an Other category. RoundImputed is used after imputation, to ensure imputed numerical values are discrete

In [75]:
class CatBinner(BaseEstimator, TransformerMixin):
    def __init__(self, cols, min_freq=0.01, fill_value="Other"):
        self.cols = cols
        self.min_freq = min_freq
        self.fill_value = fill_value

    def fit(self, X, y=None):
        self.keep_labels = {}
        for c in self.cols:
            freq = X[c].value_counts(normalize=True)
            self.keep_labels[c] = freq[freq >= self.min_freq].index.tolist()

        return self

    def transform(self, X):
        X_copy = X.copy()
        for c in self.cols:
            labels = self.keep_labels[c]
            X_copy[c] = np.where(X_copy[c].isin(labels) | X_copy[c].isna(), X_copy[c], self.fill_value)
        return X_copy

    def set_output(self, *, transform=None):
        return self

class RoundImputed(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        rounded = X[self.cols].round()
        self.min_ = rounded.min()
        self.max_ = rounded.max()
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.cols] = X_copy[self.cols].round().clip(self.min_, self.max_, axis=1)
        return X_copy

    def set_output(self, *, transform=None):
        return self

class OrdEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols, inner):
        self.cat_cols = cat_cols
        self.inner = inner

    def _encode(self, X):
        X_enc = X.copy()
        X_enc[self.cat_cols] = self.encoder_.transform(X[self.cat_cols])
        return X_enc

    def _decode(self, X_imp):
        X_dec = X_imp.copy()
        X_dec[self.cat_cols] = self.encoder_.inverse_transform(X_imp[self.cat_cols])
        return X_dec

    def fit_transform(self, X, y=None):
        self.encoder_ = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)
        self.encoder_.fit(X[self.cat_cols])
        self.inner_ = clone(self.inner)
        return self._decode(self.inner_.fit_transform(self._encode(X), y))

    def fit(self, X, y=None):
        self.fit_transform(X, y)
        return self

    def transform(self, X):
        return self._decode(self.inner_.transform(self._encode(X)))

    def set_output(self, *, transform=None):
        return self

# Building pipeline
## Define a function to construct pipeline given model, and discrete and categorical columns

In [104]:
def build_pipeline(model, cat_cols, dis_cols, scale_cols):
    imputer = IterativeImputer(estimator=HistGradientBoostingRegressor(max_iter=50, random_state=10), skip_complete=True, max_iter=5, tol=1e-2, random_state=10)

    impute = Pipeline([
        ("imputer", imputer),
        ("round", RoundImputed(cols=cat_cols + dis_cols)),
    ])

    impute.set_output(transform="pandas")

    encode = ColumnTransformer([("encode", OneHotEncoder(handle_unknown="ignore"), cat_cols), ("scale", StandardScaler(), scale_cols)], remainder="passthrough", verbose_feature_names_out=False)

    pipeline = Pipeline([
        ("bin", CatBinner(cols=cat_cols, min_freq=0.01, fill_value="Other")),
        ("imputer", OrdEncoder(cat_cols, impute)),
        ("encode", encode),
        ("model", model)
    ], memory="cache_dir")

    return pipeline

# Modeling
## Logistic regression

In [106]:
lr = LogisticRegression(l1_ratio=1, C = 0.012742749857031334, max_iter = 1000, random_state=10, solver="saga")

pipeline = build_pipeline(lr, cat_cols, dis_cols, scale_cols)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

cross_val_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=["accuracy", "balanced_accuracy", "precision", "recall", "roc_auc", "f1"], return_train_score=True, n_jobs=1, error_score="raise")



C:\Users\gelet\PycharmProjects\stop-and-search\.venv\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:597: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
An ill-conditioned matrix detected: slice 0 has rcond = 9.444217028030869e-17.
  warnings.warn(
C:\Users\gelet\PycharmProjects\stop-and-search\.venv\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:199: ConvergenceWarning: lbfgs failed to converge after 999 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=999).
You might also want to scale the data as shown in:
    https:/

test_accuracy    0.944 ± 0.001
train_accuracy   0.944 ± 0.000
test_balanced_accuracy 0.537 ± 0.001
train_balanced_accuracy 0.538 ± 0.005
test_precision   0.467 ± 0.041
train_precision  0.474 ± 0.011
test_recall      0.080 ± 0.001
train_recall     0.081 ± 0.011
test_roc_auc     0.824 ± 0.008
train_roc_auc    0.826 ± 0.002
test_f1          0.137 ± 0.003
train_f1         0.139 ± 0.016


## Decision Tree

In [107]:
dt = DecisionTreeClassifier(criterion = "entropy", max_depth = 9, min_samples_leaf = 31, min_samples_split = 29, random_state=10)

pipeline = build_pipeline(DecisionTreeClassifier(max_depth=5, random_state=10), cat_cols, dis_cols, scale_cols)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

cross_val_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=["accuracy", "balanced_accuracy", "precision", "recall", "roc_auc", "f1"], return_train_score=True, n_jobs=1, error_score="raise")

for k, v in cross_val_scores.items():
    if k.startswith(("test_", "train_")):
            print(f"{k:16s} {v.mean():.3f} ± {v.std():.3f}")

test_accuracy    0.945 ± 0.000
train_accuracy   0.945 ± 0.000
test_balanced_accuracy 0.500 ± 0.001
train_balanced_accuracy 0.502 ± 0.001
test_precision   0.180 ± 0.149
train_precision  0.719 ± 0.121
test_recall      0.001 ± 0.001
train_recall     0.003 ± 0.001
test_roc_auc     0.788 ± 0.008
train_roc_auc    0.794 ± 0.002
test_f1          0.002 ± 0.002
train_f1         0.007 ± 0.002


## Random Forest

In [108]:
rf = RandomForestClassifier(max_depth=10, random_state=10, min_samples_leaf = 2, min_samples_split = 5, n_estimators=100)

pipeline = build_simple_pipeline(rf, cat_cols, dis_cols, scale_cols)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

rf_cross_val_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=["accuracy", "balanced_accuracy", "precision", "recall", "roc_auc", "f1"], return_train_score=True, n_jobs=1, error_score="raise")

for k, v in rf_cross_val_scores.items():
    if k.startswith(("test_", "train_")):
            print(f"{k} {v.mean()} ± {v.std()}")

test_accuracy    0.945 ± 0.000
train_accuracy   0.946 ± 0.000
test_balanced_accuracy 0.503 ± 0.001
train_balanced_accuracy 0.510 ± 0.002
test_precision   0.863 ± 0.117
train_precision  0.988 ± 0.007
test_recall      0.007 ± 0.001
train_recall     0.019 ± 0.004
test_roc_auc     0.830 ± 0.007
train_roc_auc    0.869 ± 0.001
test_f1          0.014 ± 0.003
train_f1         0.038 ± 0.007
